# Set New

In [ ]:
import requests
import os

grafana_url ="HERE COMES YOUR GRAFANA URL CONFIGURED"

token = os.getenv("IRIS_Alarm_Aut")


uids = [
    
    "random-cf2f4hdswq5tse"
]

headers = {"Authorization": f"Bearer {token}"}

for uid in uids:
    url = f"{grafana_url}/api/v1/provisioning/alert-rules/{uid}"
    resp = requests.delete(url, headers=headers, verify=False)   # verify=False only if self-signed SSL
    print(uid, resp.status_code, resp.text)

# Alarm Test

In [ ]:
import requests
import json
import os
# URL del endpoint
url = "XXXXX"

# Token de autenticación
token = os.getenv("IRIS_Alarm_Aut")

# Headers HTTP
headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}"
}

# Cuerpo (payload) del request
payload = {
    "orgID": 35,
    "folderUID": "witC062nk",
    "ruleGroup": "X",
    "title": "Some Other Alert v3",
    "condition": "A",
    "data": [
        {
            "refId": "A",
            "queryType": "KQL",
            "relativeTimeRange": {
                "from": 600,
                "to": 0
            },
            "datasourceUid": "RWFnXthVk",
            "model": {
                "datasource": {
                    "type": "grafana-azure-data-explorer-datasource",
                    "uid": "RWFnXthVk"
                },
                "expression": {
                    "groupBy": {"expressions": [], "type": "and"},
                    "reduce": {"expressions": [], "type": "and"},
                    "where": {"expressions": [], "type": "and"}
                },
                "hide": False,
                "intervalMs": 1000,
                "maxDataPoints": 43200,
                "pluginVersion": "7.1.0",
                "query": "",
                "querySource": "raw",
                "queryType": "KQL",
                "rawMode": False,
                "refId": "A",
                "resultFormat": "table"
            }
        }
    ],
    "updated": "2025-10-20T11:50:42Z",
    "noDataState": "NoData",
    "execErrState": "Error",
    "for": "10m",
    "isPaused": False,
    "notification_settings": {
        "receiver": "Tom"
    }
}

# Send the POST request
response = requests.post(url, headers=headers, data=json.dumps(payload))

# Confirmation Print
print("Status Code:", response.status_code)
print("Response:")
print(response.text)


# Delete Alarms

In [ ]:
import requests
import os

grafana_url ="XXX"

token = os.getenv("IRIS_Alarm_Aut")


uids = [
    
    "af2fsjd2hfny8e"
]

headers = {"Authorization": f"Bearer {token}"}

for uid in uids:
    url = f"{grafana_url}/api/v1/provisioning/alert-rules/{uid}"
    resp = requests.delete(url, headers=headers, verify=False)   # verify=False only if self-signed SSL
    print(uid, resp.status_code, resp.text)

# Update a Panel

Builds a valid Grafana dashboard wrapper around them on the fly

In [ ]:
import os
import json
from datetime import datetime
import requests

# ==============================================================================
# ====== CONFIGURATION ======
# ==============================================================================
# The base URL of your target server instance (e.g., "https://grafana.example.com")
BASE_URL = "https://your-server-instance.com"

# Sensitive authentication token fetched directly from local environment variables
API_TOKEN = os.getenv("API_AUTHORIZATION_TOKEN")

# File path referencing the raw source configuration or fragment JSON
INPUT_FILE_PATH = r"path/to/your/input_data.json"

# Placement target reference (Set to unique ID string, or None for default root placement)
TARGET_FOLDER_ID = None  

# Target overwrite configurations for updating existing entities
TARGET_UNIQUE_ID = None
ALLOW_OVERWRITE = False

# Fallback identity markers to inject when missing from the input data source
FALLBACK_COMPONENT_TYPE = "generic-datasource-type"
FALLBACK_COMPONENT_ID   = "GENERIC_COMPONENT_UID"
DEFAULT_ENTITY_TITLE    = "Automated Resource Import"

# ==============================================================================
# ====== VALIDATION & HEADER INITIALIZATION ======
# ==============================================================================
assert API_TOKEN, "Security Halt: Please define the 'API_AUTHORIZATION_TOKEN' environment variable."

API_HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

def is_valid_payload_structure(data):
    """Evaluates if the parsed file already mirrors a complete structural schema."""
    return isinstance(data, dict) and (
        "dashboard" in data or "panels" in data or ("spec" in data and "panels" in data["spec"])
    )

def sanitize_and_apply_defaults(panel_data):
    """
    Validates, duplicates, and sanitizes an extracted object fragment.
    Ensures structural integrity by overlaying standardized fallbacks.
    """
    sanitized = dict(panel_data)  # Decouple via shallow copy
    sanitized.setdefault("id", 1)
    sanitized.setdefault("type", "timeseries")
    sanitized.setdefault("title", sanitized.get("title", f"{DEFAULT_ENTITY_TITLE} Panel"))
    sanitized.setdefault("gridPos", {"h": 8, "w": 12, "x": 0, "y": 0})
    
    # Enforce standard object map for targeted data definitions
    datasource_spec = sanitized.get("datasource")
    if isinstance(datasource_spec, str) or not datasource_spec:
        sanitized["datasource"] = {"type": FALLBACK_COMPONENT_TYPE, "uid": FALLBACK_COMPONENT_ID}
        
    # Build proper target list structure if absent or malformed
    if "targets" not in sanitized or not isinstance(sanitized["targets"], list):
        inner_model = (
            sanitized.get("model") 
            if "model" in sanitized 
            else (sanitized if ("query" in sanitized or "queryType" in sanitized) else None)
        )
        if inner_model:
            if isinstance(inner_model.get("datasource"), str) or not inner_model.get("datasource"):
                inner_model["datasource"] = {"type": FALLBACK_COMPONENT_TYPE, "uid": FALLBACK_COMPONENT_ID}
            inner_model.setdefault("refId", "A")
            sanitized["targets"] = [inner_model]
            sanitized.pop("model", None)
        else:
            sanitized["targets"] = []
            
    sanitized.setdefault("fieldConfig", {"defaults": {}, "overrides": []})
    sanitized.setdefault("options", {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "single"}})
    sanitized.setdefault("pluginVersion", "12.2.0")
    return sanitized

def standardize_payload_wrapper(raw_object):
    """
    Ingests polymorphic JSON input types (e.g., full maps, partial specs, lists, 
    or individual query objects) and unifies them into a valid endpoint schema.
    """
    # Pattern A: Object encapsulates an explicit root wrapper map
    if "dashboard" in raw_object and isinstance(raw_object["dashboard"], dict):
        structured_target = raw_object["dashboard"]
        structured_target.setdefault("panels", [])
        structured_target["panels"] = [sanitize_and_apply_defaults(p) for p in structured_target["panels"]]
        structured_target.setdefault("title", structured_target.get("title", DEFAULT_ENTITY_TITLE))
        
    # Pattern B: Object structural specifications utilize 'spec' abstractions
    elif "spec" in raw_object and isinstance(raw_object["spec"], dict) and "panels" in raw_object["spec"]:
        specification_map = raw_object["spec"]
        processed_panels = [sanitize_and_apply_defaults(p) for p in specification_map.get("panels", [])]
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": specification_map.get("title", DEFAULT_ENTITY_TITLE),
            "timezone": specification_map.get("timezone", "browser"),
            "schemaVersion": specification_map.get("schemaVersion", 41),
            "version": 0,
            "time": specification_map.get("time", {"from": "now-1h", "to": "now"}),
            "panels": processed_panels,
        }
        
    # Pattern C: Object root payload is an explicit list of array objects
    elif "panels" in raw_object and isinstance(raw_object["panels"], list):
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": raw_object.get("title", f"{DEFAULT_ENTITY_TITLE} {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC"),
            "timezone": raw_object.get("timezone", "browser"),
            "schemaVersion": raw_object.get("schemaVersion", 41),
            "version": 0,
            "time": raw_object.get("time", {"from": "now-1h", "to": "now"}),
            "panels": [sanitize_and_apply_defaults(p) for p in raw_object["panels"]],
        }
        
    # Pattern D: Fallback. Indigestible object structure assumed as raw isolated panel definition
    else:
        isolated_panel = sanitize_and_apply_defaults(raw_object)
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": f"{DEFAULT_ENTITY_TITLE} – Raw Target ({datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC)",
            "timezone": "browser",
            "schemaVersion": 41,
            "version": 0,
            "time": {"from": "now-1h", "to": "now"},
            "panels": [isolated_panel],
        }

    if TARGET_UNIQUE_ID is not None:
        structured_target["uid"] = TARGET_UNIQUE_ID
    structured_target.setdefault("schemaVersion", 41)
    
    return {
        "dashboard": structured_target,
        "folderUid": TARGET_FOLDER_ID,
        "message": "Automated pipeline processing wrapper update executed.",
        "overwrite": bool(ALLOW_OVERWRITE),
    }

# ==============================================================================
# ====== PIPELINE EXECUTION ======
# ==============================================================================
if __name__ == "__main__":
    # Load document payload
    with open(INPUT_FILE_PATH, "r", encoding="utf-8") as file_stream:
        parsed_json_input = json.load(file_stream)

    # Standardize data structure layout
    processed_payload = standardize_payload_wrapper(parsed_json_input)

    # Dispatch to remote server API
    target_endpoint = f"{BASE_URL}/api/dashboards/db"
    response = requests.post(target_endpoint, headers=API_HEADERS, json=processed_payload, timeout=60)
    
    print("Network Transaction Status:", response.status_code)
    print("Network Transaction Body Summary:", response.text)

    try:
        api_response_body = response.json()
        if response.ok and isinstance(api_response_body, dict) and "url" in api_response_body:
            print("\nTransaction Complete. Deployed location tracking reference:")
            print(BASE_URL + api_response_body["url"])
    except (json.JSONDecodeError, KeyError, TypeError):
        pass

This script is an automated pipeline designed to take raw data configurations or visualization definitions and import them onto a remote monitoring server (via an API).

In [ ]:
import os
import json
from datetime import datetime
import requests

# ==============================================================================
# ====== CONFIGURATION ======
# ==============================================================================
# The base URL of your remote server instance
BASE_URL = "https://grafana.yourdomain.com"

# Sensitive authentication token pulled securely from local environment variables
API_TOKEN = os.getenv("GRAFANA_API_TOKEN")

# Generic path placeholder replacing specific local user directories
INPUT_FILE_PATH = r"C:\path\to\your\data_pipeline\input_file.json"

# Target location reference (Set to a UID string, or None for default root placement)
TARGET_FOLDER_ID = None  

# Target override behaviors for updating pre-existing entities
TARGET_UNIQUE_ID = None
ALLOW_OVERWRITE = False

# Fallback identity specifications for empty or mismatched data fields
FALLBACK_COMPONENT_TYPE = "generic-datasource-type"
FALLBACK_COMPONENT_ID   = "GENERIC_COMPONENT_UID"
DEFAULT_ENTITY_TITLE    = "Data Automation Import"

# ==============================================================================
# ====== VALIDATION & HEADER INITIALIZATION ======
# ==============================================================================
assert API_TOKEN, "Security Halt: Please set the 'GRAFANA_API_TOKEN' environment variable."

API_HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

def is_valid_payload_structure(data):
    """Evaluates if the raw object matches a standard schema structure."""
    return isinstance(data, dict) and (
        "dashboard" in data or "panels" in data or ("spec" in data and "panels" in data["spec"])
    )

def sanitize_and_apply_defaults(panel_data):
    """
    Duplicates and sanitizes an incoming component object.
    Applies standard structures and defaults if key properties are missing.
    """
    sanitized = dict(panel_data)  # Break reference link via shallow copy
    sanitized.setdefault("id", 1)
    sanitized.setdefault("type", "timeseries")
    sanitized.setdefault("title", sanitized.get("title", f"{DEFAULT_ENTITY_TITLE} Panel"))
    sanitized.setdefault("gridPos", {"h": 8, "w": 12, "x": 0, "y": 0})
    
    # Enforce standard object mapping for target analytical definitions
    datasource_spec = sanitized.get("datasource")
    if isinstance(datasource_spec, str) or not datasource_spec:
        sanitized["datasource"] = {"type": FALLBACK_COMPONENT_TYPE, "uid": FALLBACK_COMPONENT_ID}
        
    # Standardize underlying logic query elements to match target array expectations
    if "targets" not in sanitized or not isinstance(sanitized["targets"], list):
        inner_model = (
            sanitized.get("model") 
            if "model" in sanitized 
            else (sanitized if ("query" in sanitized or "queryType" in sanitized) else None)
        )
        if inner_model:
            if isinstance(inner_model.get("datasource"), str) or not inner_model.get("datasource"):
                inner_model["datasource"] = {"type": FALLBACK_COMPONENT_TYPE, "uid": FALLBACK_COMPONENT_ID}
            inner_model.setdefault("refId", "A")
            sanitized["targets"] = [inner_model]
            sanitized.pop("model", None)
        else:
            sanitized["targets"] = []
            
    sanitized.setdefault("fieldConfig", {"defaults": {}, "overrides": []})
    sanitized.setdefault("options", {"legend": {"displayMode": "list", "placement": "bottom"}, "tooltip": {"mode": "single"}})
    sanitized.setdefault("pluginVersion", "12.2.0")
    return sanitized

def standardize_payload_wrapper(raw_object):
    """
    Ingests dynamic variations of structural configurations (full payloads, single items,
    or standalone logic fragments) and wraps them cleanly into a compliant endpoint scheme.
    """
    # Type 1: Input contains a fully formed explicit wrapper root map
    if "dashboard" in raw_object and isinstance(raw_object["dashboard"], dict):
        structured_target = raw_object["dashboard"]
        structured_target.setdefault("panels", [])
        structured_target["panels"] = [sanitize_and_apply_defaults(p) for p in structured_target["panels"]]
        structured_target.setdefault("title", structured_target.get("title", DEFAULT_ENTITY_TITLE))
        
    # Type 2: Input relies on unified modern architectural data design specifications ('spec')
    elif "spec" in raw_object and isinstance(raw_object["spec"], dict) and "panels" in raw_object["spec"]:
        specification_map = raw_object["spec"]
        processed_panels = [sanitize_and_apply_defaults(p) for p in specification_map.get("panels", [])]
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": specification_map.get("title", DEFAULT_ENTITY_TITLE),
            "timezone": specification_map.get("timezone", "browser"),
            "schemaVersion": specification_map.get("schemaVersion", 41),
            "version": 0,
            "time": specification_map.get("time", {"from": "now-1h", "to": "now"}),
            "panels": processed_panels,
        }
        
    # Type 3: Input is a traditional document structural array map containing root panels
    elif "panels" in raw_object and isinstance(raw_object["panels"], list):
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": raw_object.get("title", f"{DEFAULT_ENTITY_TITLE} {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC"),
            "timezone": raw_object.get("timezone", "browser"),
            "schemaVersion": raw_object.get("schemaVersion", 41),
            "version": 0,
            "time": raw_object.get("time", {"from": "now-1h", "to": "now"}),
            "panels": [sanitize_and_apply_defaults(p) for p in raw_object["panels"]],
        }
        
    # Type 4: Raw alternative fallback. Treats the entire input config as an isolated visual element
    else:
        isolated_panel = sanitize_and_apply_defaults(raw_object)
        structured_target = {
            "id": None,
            "uid": TARGET_UNIQUE_ID,
            "title": f"{DEFAULT_ENTITY_TITLE} – Automated Wrapper ({datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC)",
            "timezone": "browser",
            "schemaVersion": 41,
            "version": 0,
            "time": {"from": "now-1h", "to": "now"},
            "panels": [isolated_panel],
        }

    if TARGET_UNIQUE_ID is not None:
        structured_target["uid"] = TARGET_UNIQUE_ID
    structured_target.setdefault("schemaVersion", 41)
    
    return {
        "dashboard": structured_target,
        "folderUid": TARGET_FOLDER_ID,
        "message": "Automated import wrapper deployment successful.",
        "overwrite": bool(ALLOW_OVERWRITE),
    }

# ==============================================================================
# ====== PIPELINE EXECUTION ======
# ==============================================================================
if __name__ == "__main__":
    # Ingest file schema data
    with open(INPUT_FILE_PATH, "r", encoding="utf-8") as file_stream:
        parsed_json_input = json.load(file_stream)

    # Convert structural layout to destination schema conventions
    processed_payload = standardize_payload_wrapper(parsed_json_input)

    # Deliver standardized configuration metadata to server API
    target_endpoint = f"{BASE_URL}/api/dashboards/db"
    response = requests.post(target_endpoint, headers=API_HEADERS, json=processed_payload, timeout=60)
    
    print("API Response Status Code:", response.status_code)
    print("API Response Content:", response.text)

    try:
        api_response_body = response.json()
        if response.ok and isinstance(api_response_body, dict) and "url" in api_response_body:
            print("\nProcessing complete! Direct monitoring link resource:")
            print(BASE_URL + api_response_body["url"])
    except (json.JSONDecodeError, KeyError, TypeError):
        pass

# Kleine Anpassungen - Overwrite a Panel

In [ ]:
import json
import uuid
import re
from pathlib import Path

def create_resource_configuration(
    template_path: str,
    old_identifier: str,
    new_identifier: str,
    output_directory: str,
    target_datasource_uid: str = "GENERIC_DATASOURCE_UID"
):
    """
    Reads a master dashboard configuration JSON template and generates a localized 
    variant dedicated to a specific targeting resource string.

    - Replaces unique target component naming conventions across string blocks and query texts.
    - Swaps out dynamic platform resource mapping IDs.
    - Generates a fresh, unique tracking UUID for the newly minted layout configuration asset.
    - Writes the standardized config file into an output storage directory.
    """

    # Handle explicit character mapping variations (e.g., standard vs. sanitized text formatting)
    old_sanitized_id = old_identifier.replace("+", "_plus_")
    new_sanitized_id = new_identifier.replace("+", "_plus_")

    # Ingest baseline structural map
    with open(template_path, "r", encoding="utf-8") as file_stream:
        configuration_data = json.load(file_stream)

    # Convert mapping dictionary into a raw string blob for global pattern replacement
    json_string_payload = json.dumps(configuration_data, indent=2, ensure_ascii=False)

    # --- 1️⃣ Apply Core Global Text Replacements ---
    json_string_payload = json_string_payload.replace(old_identifier, new_identifier)
    json_string_payload = json_string_payload.replace(old_sanitized_id, new_sanitized_id)

    # --- 2️⃣ Align Target Backend Component Connections ---
    json_string_payload = re.sub(
        r'"datasource":\s*\{\s*"type":\s*"(grafana[^"]+)"\s*,\s*"uid":\s*"[a-f0-9\-]+"',
        rf'"datasource": {{ "type": "\1", "uid": "{target_datasource_uid}"',
        json_string_payload
    )

    # --- 3️⃣ Inject Unique Reference Identity Token ---
    new_asset_uuid = str(uuid.uuid4())
    json_string_payload = re.sub(
        r'"uid":\s*"[a-zA-Z0-9\-_]+"',
        f'"uid": "{new_asset_uuid}"',
        json_string_payload,
        count=1  # Target strictly the primary outer configuration entity identifier
    )

    # --- 4️⃣ Build Output Directories & Persist Generated File ---
    Path(output_directory).mkdir(parents=True, exist_ok=True)
    generated_filename = f"{new_identifier}.json"
    destination_file_path = Path(output_directory) / generated_filename

    with open(destination_file_path, "w", encoding="utf-8") as output_stream:
        output_stream.write(json_string_payload)

    print(f"✅ Success: Generated configuration at: {destination_file_path}")
    print(f"🆕 New Managed Entity Unique UID: {new_asset_uuid}")
    return destination_file_path


def batch_generate_resource_profiles(resource_list, template_path, reference_id, output_directory, target_datasource_uid="GENERIC_DATASOURCE_UID"):
    """
    Loops through an array of dynamic configuration target identifiers to batch-compile 
    individual configuration dashboard structures based on a single master layout template.
    """
    for entry in resource_list:
        # Normalize elements wrapped in single-item lists if passing data tables directly
        target_name = entry[0] if isinstance(entry, list) else entry
        
        create_resource_configuration(
            template_path=template_path,
            old_identifier=reference_id,
            new_identifier=target_name,
            output_directory=output_directory,
            target_datasource_uid=target_datasource_uid
        )


# ==============================================================================
# ====== ENTRY EXECUTION ======
# ==============================================================================
if __name__ == "__main__":
    # Populated array specifying target profiles to build dynamically
    target_resources_to_generate = [
        "RESOURCE_ALPHA_01",
        "RESOURCE_ALPHA_02",
        "RESOURCE_ALPHA_03",
        "RESOURCE_BETA_01",
        "RESOURCE_BETA_02"
    ]

    batch_generate_resource_profiles(
        resource_list=target_resources_to_generate,
        template_path=r"C:\workspace\data_templates\master_dashboard_blueprint.json",
        reference_id="BASE_TEMPLATE_IDENTIFIER",
        output_directory=r"C:\workspace\data_templates\output_batch_run",
        target_datasource_uid="GENERIC_DATASOURCE_UID"
    )